In [6]:
import numpy as np
import matplotlib.pyplot as plt
import pymcel as pc

Bienvenido a PyMCel v0.9.17 ¡al infinito y más allá!


In [3]:
def pendulo_elastico(Y, t, L, k, m):

    g = 9.81 # m/s^2

    #Variables generalizadas
    q1, q2, q1_punto, q2_punto = Y

    dq1_dt = q1_punto
    dq2_dt = q2_punto

    #aceleraciones 

    dq1_punto_dt = (- g * np.sin(q1) - 2 * q1_punto * q2_punto / (L + q2))
    dq2_punto_dt = (L + q2) * q1_punto ** 2 + g * np.cos(q1) - (k/m) * q2

    return [dq1_dt, dq2_dt, dq1_punto_dt, dq2_punto_dt]



In [32]:
# Condiciones iniciales
L = 1
k = 10
m = 1

y = [np.pi / 4, 0.1, 0, 0]  # [q1, q2, q1_punto, q2_punto]
# Parte de un ángulo de pi/4 con una elongación de 0.1 y con velocidad 0 

ts = np.linspace(0, 10, 150)

from scipy.integrate import odeint
solucion = odeint(pendulo_elastico, y, ts, args = (L, k, m))

q1s = solucion[:,0]
q2s = solucion[:,1]

q1s, q2s

(array([ 0.78539816,  0.7699418 ,  0.72551011,  0.65748952,  0.57346192,
         0.48141844,  0.38814012,  0.29837954,  0.21490525,  0.1390171 ,
         0.07112544,  0.01119346, -0.04098644, -0.08564098, -0.12293932,
        -0.15295472, -0.17564101, -0.19080987, -0.19809935, -0.19692585,
        -0.18641191, -0.1652822 , -0.13172683, -0.08325627, -0.0166517 ,
         0.0716878 ,  0.1843593 ,  0.31980352,  0.46825779,  0.61140116,
         0.72952911,  0.81043539,  0.85144921,  0.85581602,  0.8289168 ,
         0.77640063,  0.70367954,  0.61592719,  0.51810491,  0.41486397,
         0.31035744,  0.20806287,  0.11069782,  0.02025054, -0.06190268,
        -0.13483931, -0.19792836, -0.25068228, -0.29262443, -0.3231625 ,
        -0.34145018, -0.34621257, -0.33550199, -0.30634512, -0.25426438,
        -0.1728014 , -0.05372788,  0.10994459,  0.31496922,  0.53687693,
         0.73658168,  0.88609331,  0.97937813,  1.02257843,  1.023963  ,
         0.99026175,  0.9265136 ,  0.83697744,  0.7

In [33]:
# Graficar la solución y crear la animación
import plotly.graph_objects as go
import numpy as np

# Transformar a coordenadas cartesianas
xs = (L + q2s) * np.sin(q1s)
ys = -(L + q2s) * np.cos(q1s)

# Función paramétrica para dibujar un resorte
def get_spring_coords(x_end, y_end, n_coils=10, width=0.15):
    length = np.sqrt(x_end**2 + y_end**2)
    if length == 0:
        return [0, 0], [0, 0]
    
    # Vectores unitarios a lo largo del resorte y perpendicular a este
    u = np.array([x_end, y_end]) / length
    n = np.array([u[1], -u[0]])
    
    t = np.linspace(0, 1, 200)
    # Envolvente para que los extremos del resorte converjan al origen y a la masa limpia y suavemente
    envelope = np.sin(np.pi * t)**0.3
    offset = width * np.sin(2 * np.pi * n_coils * t) * envelope
    
    x_spring = t * x_end + offset * n[0]
    y_spring = t * y_end + offset * n[1]
    return x_spring, y_spring

# Calcular límites para los ejes
rmax = L + max(q2s) + 0.5
x_lim = [-rmax, rmax]
y_lim = [-rmax, 0.5]

x0_spring, y0_spring = get_spring_coords(xs[0], ys[0])

# Crear la figura dividiendo el gráfico en 2 capas (trazos): el resorte y la masa
fig = go.Figure(
    data=[
        go.Scatter(
            x=x0_spring, 
            y=y0_spring, 
            mode="lines", 
            line=dict(color="black", width=2),
            name="Resorte"
        ),
        go.Scatter(
            x=[xs[0]], 
            y=[ys[0]], 
            mode="markers", 
            marker=dict(size=22, color="blue"),
            name="Masa"
        )
    ],
    layout=go.Layout(
        title="Animación del Péndulo Elástico",
        xaxis=dict(range=x_lim, scaleanchor="y", scaleratio=1),
        yaxis=dict(range=y_lim),
        width=500,
        height=500,
        showlegend=False,
        updatemenus=[dict(
            type="buttons",
            showactive=False,
            buttons=[
                dict(label="Play",
                     method="animate",
                     args=[None, {"frame": {"duration": 50, "redraw": False}, "fromcurrent": True}]),
                dict(label="Pause",
                     method="animate",
                     args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}])
            ]
        )]
    )
)

# Crear los frames para la animación
frames = []
for i in range(len(ts)):
    x_sp, y_sp = get_spring_coords(xs[i], ys[i])
    frames.append(go.Frame(
        data=[
            go.Scatter(x=x_sp, y=y_sp),
            go.Scatter(x=[xs[i]], y=[ys[i]])
        ],
        name=str(i)
    ))
    
fig.frames = frames

fig.show()